# MCDE Single Final Pipeline

This is the one notebook to run from now on. It consolidates the final proof layer, real multi-decision training outputs, safety-car/risk training, steady deep audits, and final display report.

It does **not** rerun raw-data ingestion. It skips completed stages unless force flags are enabled. Deep audit is bounded and steady, not an infinite or memory-heavy loop.


## 1. Mount Drive

Colab must mount Drive. All code is locked to `/content/drive/MyDrive/ibm_project_stuff/MDCE`.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 2. Install Dependencies


In [ ]:
%pip install -q pandas numpy pyarrow scikit-learn matplotlib seaborn joblib


## 3. Run Final Pipeline

This one cell runs all missing final stages and deep audit. It is designed to skip work already completed.


In [ ]:

from pathlib import Path
import json
import os
import traceback

ROOT = Path(os.environ.get("MDCE_ROOT", "/content/drive/MyDrive/ibm_project_stuff/MDCE"))
if not ROOT.exists():
    raise FileNotFoundError(f"Expected project folder not found: {ROOT}. This notebook is locked to the MDCE folder.")

REPORT_DIR = ROOT / "outputs" / "reports"
MODEL_DIR = ROOT / "outputs" / "models"
CHART_DIR = ROOT / "outputs" / "charts"
for folder in [REPORT_DIR, MODEL_DIR, CHART_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Safe defaults: no raw ingestion, no rerun of completed heavy stages.
# Set a force flag to True only when you intentionally want that stage repeated.
FORCE_FINALIZE_PIT = False
FORCE_RETRAIN_REAL_MULTIDECISION = False
FORCE_RETRAIN_SAFETY_CAR_RISK = False
FORCE_DEEP_AUDIT = False

# Deep audit is steady, not infinite. It runs bounded repeated-seed and bootstrap checks.
RUN_DEEP_AUDIT = True

PIPELINE_ERRORS = []

def run_stage(stage_name, source, required_outputs=None, force=False, allow_blocked=True):
    required_outputs = [Path(path) for path in (required_outputs or [])]
    outputs_exist = required_outputs and all(path.exists() for path in required_outputs)
    if outputs_exist and not force:
        print(f"SKIP {stage_name}: outputs already exist")
        for path in required_outputs:
            print(" -", path)
        return "skipped_existing"
    print(f"RUN {stage_name}")
    try:
        exec(source, globals())
        return "completed"
    except RuntimeError as exc:
        message = str(exc)
        if allow_blocked and message.startswith("blocked"):
            print(f"BLOCKED {stage_name}:", message)
            PIPELINE_ERRORS.append({"stage": stage_name, "type": "blocked", "message": message})
            return "blocked"
        print(f"FAILED {stage_name}:", message)
        traceback.print_exc()
        PIPELINE_ERRORS.append({"stage": stage_name, "type": "runtime_error", "message": message})
        raise
    except Exception as exc:
        print(f"FAILED {stage_name}:", exc)
        traceback.print_exc()
        PIPELINE_ERRORS.append({"stage": stage_name, "type": "exception", "message": str(exc)})
        raise


# Always prefer running the latest on-Drive python scripts over the embedded
# notebook snapshots below. This keeps the notebook in sync with script fixes.

# Embedded stage-source snapshots removed to avoid drift.
# This notebook loads the latest versions from python scripts in ROOT/notebooks/.
def load_stage_source(script_name):
    path = ROOT / "notebooks" / script_name
    if not path.exists():
        raise FileNotFoundError(f"Missing stage script: {path}")
    return path.read_text(encoding="utf-8")

PIT_FINALIZER_SOURCE = load_stage_source("mdce_pit_proof_finalizer.py")
REAL_MULTIDECISION_SOURCE = load_stage_source("mdce_real_multidecision_training.py")
SAFETY_CAR_RISK_SOURCE = load_stage_source("mdce_safety_car_risk_training.py")
DEEP_AUDIT_SOURCE = load_stage_source("mdce_deep_audit_workload.py")
FINAL_CONSOLIDATOR_SOURCE = load_stage_source("mdce_final_display_consolidator.py")

stage_results = {}
stage_results["pit_finalizer"] = run_stage(
    "pit proof finalizer",
    PIT_FINALIZER_SOURCE,
    required_outputs=[REPORT_DIR / "pit_final_model_decision.json", REPORT_DIR / "pit_final_model_decision.md"],
    force=FORCE_FINALIZE_PIT,
)
stage_results["real_multidecision"] = run_stage(
    "real-label multi-decision training",
    REAL_MULTIDECISION_SOURCE,
    required_outputs=[REPORT_DIR / "mdce_real_multidecision_training_summary.json", REPORT_DIR / "mdce_real_decision_training_registry.csv"],
    force=FORCE_RETRAIN_REAL_MULTIDECISION,
)
stage_results["safety_car_risk"] = run_stage(
    "safety-car/risk training",
    SAFETY_CAR_RISK_SOURCE,
    required_outputs=[REPORT_DIR / "safety_car_risk_model_metrics.json", REPORT_DIR / "safety_car_risk_training.md"],
    force=FORCE_RETRAIN_SAFETY_CAR_RISK,
)
if RUN_DEEP_AUDIT:
    stage_results["deep_audit"] = run_stage(
        "steady deep audit workload",
        DEEP_AUDIT_SOURCE,
        required_outputs=[REPORT_DIR / "MCDE_DEEP_AUDIT_SUMMARY.json", REPORT_DIR / "MCDE_DEEP_AUDIT_SUMMARY.md"],
        force=FORCE_DEEP_AUDIT,
    )
else:
    stage_results["deep_audit"] = "disabled"

print("RUN final display consolidation")
exec(FINAL_CONSOLIDATOR_SOURCE, globals())

pipeline_status = {
    "stage_results": stage_results,
    "errors": PIPELINE_ERRORS,
    "final_report": str(REPORT_DIR / "MCDE_FINAL_DISPLAY_REPORT.md"),
    "final_registry": str(REPORT_DIR / "MCDE_FINAL_MODEL_REGISTRY.csv"),
    "deep_audit_report": str(REPORT_DIR / "MCDE_DEEP_AUDIT_SUMMARY.md"),
}
(REPORT_DIR / "MCDE_SINGLE_FINAL_PIPELINE_STATUS.json").write_text(json.dumps(pipeline_status, indent=2, default=str), encoding="utf-8")
print("MCDE SINGLE FINAL PIPELINE COMPLETE")
print(json.dumps(pipeline_status, indent=2))


## 4. Final Viewer


In [ ]:

from pathlib import Path
from IPython.display import Markdown, Image, display
import pandas as pd
import json
import os

ROOT = Path(os.environ.get("MDCE_ROOT", "/content/drive/MyDrive/ibm_project_stuff/MDCE"))
REPORT_DIR = ROOT / "outputs" / "reports"
CHART_DIR = ROOT / "outputs" / "charts"

final_report = REPORT_DIR / "MCDE_FINAL_DISPLAY_REPORT.md"
final_registry = REPORT_DIR / "MCDE_FINAL_MODEL_REGISTRY.csv"
deep_report = REPORT_DIR / "MCDE_DEEP_AUDIT_SUMMARY.md"
status_json = REPORT_DIR / "MCDE_SINGLE_FINAL_PIPELINE_STATUS.json"

print("Final report:", final_report)
print("Final registry:", final_registry)
print("Deep audit:", deep_report)
print("Pipeline status:", status_json)
if status_json.exists():
    print(status_json.read_text(encoding="utf-8"))
if final_report.exists():
    display(Markdown(final_report.read_text(encoding="utf-8")))
if final_registry.exists():
    display(pd.read_csv(final_registry))
if deep_report.exists():
    display(Markdown(deep_report.read_text(encoding="utf-8")))

charts = [
    "pit_model_confusion_matrix.png",
    "model_challenger_average_precision.png",
    "real_multidecision_confusion_matrices.png",
    "stint_remaining_regression_holdout.png",
    "safety_car_risk_confusion_matrix.png",
    "safety_car_risk_leaderboard.png",
    "deep_pit_seed_stability.png",
    "deep_pit_bootstrap_f1.png",
    "deep_stint_remaining_seed_stability.png",
]
for chart in charts:
    path = CHART_DIR / chart
    print("Chart:", path, "exists:", path.exists())
    if path.exists():
        display(Image(filename=str(path)))


## 5. What To Send Back

Send only this after the run:

- The `MCDE SINGLE FINAL PIPELINE COMPLETE` block.
- The displayed `MCDE_FINAL_DISPLAY_REPORT.md`.
- The displayed `MCDE_FINAL_MODEL_REGISTRY.csv`.
- The displayed `MCDE_DEEP_AUDIT_SUMMARY.md`.
- Any traceback if the notebook stops.
